# Project 01 — Churn Prediction

Predicting bank customer churn using multiple approaches:
baseline classifiers, Random Forest, Scikit-learn MLP, Keras, TensorFlow, and a custom NumPy neural network.

In [1]:
%pip install tensorflow scikit-learn numpy pandas --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
os.makedirs("../models", exist_ok=True)

In [ ]:
import sys, os
sys.path.insert(0, '../src')
sys.path.insert(0, '..')  # for my_neural_net.py
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"

from features import get_train_test_data
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

data = get_train_test_data(
    "../data/bank_data_train.csv",
    num_impute="mean",
)

print(f"x_train: {data.x_train.shape}")
print(f"x_test:  {data.x_test.shape}")
data.x_train.head()


x_train: (284152, 113)
x_test:  (71038, 113)


,CR_PROD_CNT_IL,AMOUNT_RUB_CLO_PRC,PRC_ACCEPTS_A_EMAIL_LINK,APP_REGISTR_RGN_CODE,PRC_ACCEPTS_A_POS,PRC_ACCEPTS_A_TK,TURNOVER_DYNAMIC_IL_1M,CNT_TRAN_AUT_TENDENCY1M,SUM_TRAN_AUT_TENDENCY1M,AMOUNT_RUB_SUP_PRC,...,PACK_103,PACK_104,PACK_105,PACK_107,PACK_108,PACK_109,PACK_301,PACK_K01,PACK_M01,PACK_O01
54105,2.720640,0.023391,0.0,0.302862,0.0,0.0,-0.047265,-1.067741,-1.651879,1.976389,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
313302,-0.274469,-0.479320,0.0,0.091351,0.0,0.0,-0.047265,0.050608,0.054461,-0.556620,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
302648,-0.274469,-0.479320,0.0,0.091351,0.0,0.0,-0.047265,0.050608,0.054461,-0.708387,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
350887,2.720640,0.336979,0.0,0.302862,0.0,0.0,-0.047265,0.050608,0.054461,0.505013,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
41493,-0.274469,-0.423432,0.0,0.302862,0.0,0.0,-0.047265,3.576069,3.315444,0.771324,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
print(f"Features used for training: {data.x_train.columns.tolist()}")

Features used for training: ['CR_PROD_CNT_IL', 'AMOUNT_RUB_CLO_PRC', 'PRC_ACCEPTS_A_EMAIL_LINK', 'APP_REGISTR_RGN_CODE', 'PRC_ACCEPTS_A_POS', 'PRC_ACCEPTS_A_TK', 'TURNOVER_DYNAMIC_IL_1M', 'CNT_TRAN_AUT_TENDENCY1M', 'SUM_TRAN_AUT_TENDENCY1M', 'AMOUNT_RUB_SUP_PRC', 'PRC_ACCEPTS_A_AMOBILE', 'SUM_TRAN_AUT_TENDENCY3M', 'PRC_ACCEPTS_TK', 'PRC_ACCEPTS_A_MTP', 'REST_DYNAMIC_FDEP_1M', 'CNT_TRAN_AUT_TENDENCY3M', 'CNT_ACCEPTS_TK', 'REST_DYNAMIC_SAVE_3M', 'CR_PROD_CNT_VCU', 'REST_AVG_CUR', 'CNT_TRAN_MED_TENDENCY1M', 'AMOUNT_RUB_NAS_PRC', 'TRANS_COUNT_SUP_PRC', 'CNT_TRAN_CLO_TENDENCY1M', 'SUM_TRAN_MED_TENDENCY1M', 'PRC_ACCEPTS_A_ATM', 'PRC_ACCEPTS_MTP', 'TRANS_COUNT_NAS_PRC', 'CNT_ACCEPTS_MTP', 'CR_PROD_CNT_TOVR', 'CR_PROD_CNT_PIL', 'SUM_TRAN_CLO_TENDENCY1M', 'TURNOVER_CC', 'TRANS_COUNT_ATM_PRC', 'AMOUNT_RUB_ATM_PRC', 'TURNOVER_PAYM', 'AGE', 'CNT_TRAN_MED_TENDENCY3M', 'CR_PROD_CNT_CC', 'SUM_TRAN_MED_TENDENCY3M', 'REST_DYNAMIC_FDEP_3M', 'REST_DYNAMIC_IL_1M', 'SUM_TRAN_CLO_TENDENCY3M', 'LDEAL_TENOR_M

## Class Imbalance Handling

The dataset is heavily imbalanced (~91 % non-churn / ~9 % churn).
Balanced class weights are computed here and passed to all neural-network `fit()` calls so that the minority class receives proportionally higher loss weighting — no resampling required.


In [4]:
print("Class distribution (train):")
print(data.y_train.value_counts().to_string())
print(f"\nBalanced class weights: {data.class_weight}")


Class distribution (train):
TARGET
0    261012
1     23140

Balanced class weights: {0: 0.5443274638713929, 1: 6.139844425237683}


## 1. Baseline Solutions

### 1.1 Naive Classifier

Always predicts the most frequent class.

In [5]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

naive_clf = DummyClassifier(strategy='most_frequent')
naive_clf.fit(data.x_train, data.y_train)

naive_y_pred  = naive_clf.predict(data.x_test)
naive_y_proba = naive_clf.predict_proba(data.x_test)[:, 1]

naive_acc = accuracy_score(data.y_test, naive_y_pred)
naive_auc = roc_auc_score(data.y_test, naive_y_proba)

print(f"Accuracy: {naive_acc:.4f}")
print(f"AUC:      {naive_auc:.4f}")
print()
print(classification_report(data.y_test, naive_y_pred))

Accuracy: 0.9186
AUC:      0.5000

              precision    recall  f1-score   support

           0       0.92      1.00      0.96     65253
           1       0.00      0.00      0.00      5785

    accuracy                           0.92     71038
   macro avg       0.46      0.50      0.48     71038
weighted avg       0.84      0.92      0.88     71038



/goinfre/stamim/churn/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/goinfre/stamim/churn/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/goinfre/stamim/churn/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", res

### 1.2 Random Forest with Grid Search

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import GridSearchCV

rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt'],
}

rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=rf_param_grid,
    scoring='roc_auc',
    cv=3,
    verbose=1,
    n_jobs=-1,
)
rf_grid.fit(data.x_train, data.y_train)

rf_best_params = rf_grid.best_params_
rf_auc = roc_auc_score(data.y_test, rf_grid.predict_proba(data.x_test)[:, 1])
rf_acc = accuracy_score(data.y_test, rf_grid.predict(data.x_test))

print(f"Best params: {rf_best_params}")
print(f"AUC:         {rf_auc:.4f}")
print(f"Accuracy:    {rf_acc:.4f}")

Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best params: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
AUC:         0.8353
Accuracy:    0.9194


## 2. Neural Networks

### 2.1 NumPy Neural Network

Custom fully-connected neural network implemented from scratch using NumPy (OOP, matrix calculations).

In [7]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), '..', 'src'))

from my_neural_net import MyNeuralNet

NUMPY_HIDDEN = [512, 256, 128]
NUMPY_EPOCHS = 60
NUMPY_LR     = 0.001
NUMPY_BATCH  = 256

numpy_model = MyNeuralNet(
    hidden_layers_size=NUMPY_HIDDEN,
    max_iter=NUMPY_EPOCHS,
    learning_rate=NUMPY_LR,
    batch_size=NUMPY_BATCH,
)
numpy_model.fit(
    data.x_train,
    data.y_train,
    class_weight=data.class_weight,
    validation_data=(data.x_test, data.y_test),
)

numpy_auc = numpy_model.auc_score(data.x_test, data.y_test)
numpy_acc = numpy_model.score(data.x_test, data.y_test)

print(f"AUC:      {numpy_auc:.4f}")
print(f"Accuracy: {numpy_acc:.4f}")


Epoch 1/60, Loss: 0.6840 - Acc: 0.5991 - AUC: 0.6628 - Val Acc: 0.5977 - Val AUC: 0.6605
Epoch 2/60, Loss: 0.6452 - Acc: 0.6222 - AUC: 0.6978 - Val Acc: 0.6205 - Val AUC: 0.6921
Epoch 3/60, Loss: 0.6286 - Acc: 0.6642 - AUC: 0.7171 - Val Acc: 0.6607 - Val AUC: 0.7096
Epoch 4/60, Loss: 0.6164 - Acc: 0.6555 - AUC: 0.7318 - Val Acc: 0.6529 - Val AUC: 0.7230
Epoch 5/60, Loss: 0.6066 - Acc: 0.6803 - AUC: 0.7419 - Val Acc: 0.6767 - Val AUC: 0.7323
Epoch 6/60, Loss: 0.5982 - Acc: 0.6599 - AUC: 0.7513 - Val Acc: 0.6557 - Val AUC: 0.7407
Epoch 7/60, Loss: 0.5909 - Acc: 0.6776 - AUC: 0.7585 - Val Acc: 0.6730 - Val AUC: 0.7471
Epoch 8/60, Loss: 0.5846 - Acc: 0.6870 - AUC: 0.7644 - Val Acc: 0.6836 - Val AUC: 0.7526
Epoch 9/60, Loss: 0.5790 - Acc: 0.6880 - AUC: 0.7697 - Val Acc: 0.6843 - Val AUC: 0.7573
Epoch 10/60, Loss: 0.5738 - Acc: 0.6851 - AUC: 0.7743 - Val Acc: 0.6809 - Val AUC: 0.7612
Epoch 11/60, Loss: 0.5696 - Acc: 0.6901 - AUC: 0.7785 - Val Acc: 0.6858 - Val AUC: 0.7650
Epoch 12/60, Loss: 

### 2.2 Scikit-learn MLPClassifier

In [8]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

MLP_HIDDEN   = (512, 256, 128)
MLP_MAX_ITER = 10
MLP_LR       = 0.001
MLP_BATCH    = 256

mlp_model = MLPClassifier(
    hidden_layer_sizes=MLP_HIDDEN,
    activation="relu",
    solver="adam",
    learning_rate_init=MLP_LR,
    max_iter=MLP_MAX_ITER,
    random_state=42,
    verbose=True,
    batch_size=MLP_BATCH,
)
mlp_model.fit(data.x_train, data.y_train)

mlp_auc = roc_auc_score(data.y_test, mlp_model.predict_proba(data.x_test)[:, 1])
mlp_acc = accuracy_score(data.y_test, mlp_model.predict(data.x_test))

print(f"AUC:      {mlp_auc:.4f}")
print(f"Accuracy: {mlp_acc:.4f}")

Iteration 1, loss = 0.24254425
Iteration 2, loss = 0.22715369
Iteration 3, loss = 0.22265403
Iteration 4, loss = 0.21970038
Iteration 5, loss = 0.21707609
Iteration 6, loss = 0.21521434
Iteration 7, loss = 0.21313747
Iteration 8, loss = 0.21091619
Iteration 9, loss = 0.20881592
Iteration 10, loss = 0.20643262


/goinfre/stamim/churn/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


AUC:      0.8294
Accuracy: 0.9174


### 2.3 Keras (TensorFlow)

Residual fully-connected network built with `tf.keras`, trained with grid search over hyperparameters.

In [10]:
from NN_GridSearch import (
    create_model,
    PARAM_GRID,
    LOSS_FUNCTION,
)
from tensorflow.keras.callbacks import EarlyStopping
from itertools import product
import pandas as pd

KERAS_MODEL_PATH = "../models/best_model_keras.keras"
KERAS_RUN_GRID = False

if KERAS_RUN_GRID:
    keras_keys = list(PARAM_GRID.keys())
    keras_combos = list(product(*PARAM_GRID.values()))
else:
    keras_combos = [(0.001, 0.1, 1e-6, 256, 200, 512)]
    keras_keys = [
        "learning_rate",
        "dropout_rate",
        "l2_reg",
        "batch_size",
        "epochs",
        "hidden_units",
    ]

keras_global_best = {"auc": 0.0}
keras_results = []
keras_params = {}


for run_idx, values in enumerate(keras_combos, 1):
    keras_params = dict(zip(keras_keys, values))
    print(f"\n[{run_idx}/{len(keras_combos)}] {keras_params}")

    tf.keras.backend.clear_session()

    keras_model = create_model(
        data.x_train.shape[1],
        keras_params["hidden_units"],
        dropout_rate=keras_params["dropout_rate"],
        l2_reg=keras_params["l2_reg"],
    )
    keras_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=keras_params["learning_rate"]),
        loss=LOSS_FUNCTION,
        metrics=[tf.keras.metrics.AUC(name="auc"), "accuracy"],
    )

    erl = EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=3,
        restore_best_weights=True,
    )

    keras_model.fit(
        data.x_train,
        data.y_train,
        epochs=keras_params["epochs"],
        batch_size=keras_params["batch_size"],
        callbacks=[erl],
        validation_data=(data.x_test, data.y_test),
        class_weight=data.class_weight,
        verbose=1,
    )
    _, keras_val_auc, _ = keras_model.evaluate(data.x_test, data.y_test, verbose=0)
    print(f"  ✔ best val_auc: {keras_val_auc:.4f}")
    keras_results.append({**keras_params, "val_auc": keras_val_auc})
    if keras_val_auc > keras_global_best["auc"]:
        keras_global_best["auc"] = keras_val_auc
        keras_model.save(KERAS_MODEL_PATH)

keras_df = (
    pd.DataFrame(keras_results)
    .sort_values("val_auc", ascending=False)
    .reset_index(drop=True)
)
keras_df.index += 1
print(f"\n{'═'*70}\n{keras_df.to_string()}\n{'═'*70}")

keras_best_model = tf.keras.models.load_model(KERAS_MODEL_PATH)
_, keras_auc, keras_acc = keras_best_model.evaluate(data.x_test, data.y_test, verbose=0)
print(f"\nKeras best model  —  test AUC: {keras_auc:.4f}  |  accuracy: {keras_acc:.4f}")



[1/1] {'learning_rate': 0.001, 'dropout_rate': 0.1, 'l2_reg': 1e-06, 'batch_size': 256, 'epochs': 200, 'hidden_units': 512}
Epoch 1/200
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 18s 14ms/step - accuracy: 0.6451 - auc: 0.7795 - loss: 0.5634 - val_accuracy: 0.6560 - val_auc: 0.8058 - val_loss: 0.5623
Epoch 2/200
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - accuracy: 0.6793 - auc: 0.8162 - loss: 0.5224 - val_accuracy: 0.6889 - val_auc: 0.8155 - val_loss: 0.5183
Epoch 3/200
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - accuracy: 0.6839 - auc: 0.8252 - loss: 0.5105 - val_accuracy: 0.6563 - val_auc: 0.8205 - val_loss: 0.5092
Epoch 4/200
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 16s 15ms/step - accuracy: 0.6926 - auc: 0.8314 - loss: 0.5023 - val_accuracy: 0.7280 - val_auc: 0.8271 - val_loss: 0.4981
Epoch 5/200
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - accuracy: 0.7019 - auc: 0.8360 - loss: 0.4962 - val_accuracy: 0.6966 - val_auc: 0.8271 - val_loss: 0.5285
Epoch 6/200
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 16s 14m

### 2.4 TensorFlow

Same residual architecture, trained independently with the TensorFlow backend.

In [11]:
TF_MODEL_PATH = "../models/best_model_tf.keras"
TF_RUN_GRID = True

if TF_RUN_GRID:
    tf_keys = list(PARAM_GRID.keys())
    tf_combos = list(product(*PARAM_GRID.values()))
else:
    tf_combos = [(0.001, 0.1, 1e-6, 256, 200, 512)]
    tf_keys = [
        "learning_rate",
        "dropout_rate",
        "l2_reg",
        "batch_size",
        "epochs",
        "hidden_units",
    ]

tf_global_best = {"auc": 0.0}
tf_results = []
tf_params = {}

for run_idx, values in enumerate(tf_combos, 1):
    tf_params = dict(zip(tf_keys, values))
    print(f"\n[{run_idx}/{len(tf_combos)}] {tf_params}")

    tf.keras.backend.clear_session()

    tf_model = create_model(
        data.x_train.shape[1],
        tf_params["hidden_units"],
        dropout_rate=tf_params["dropout_rate"],
        l2_reg=tf_params["l2_reg"],
    )
    tf_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=tf_params["learning_rate"]),
        loss=LOSS_FUNCTION,
        metrics=[tf.keras.metrics.AUC(name="auc"), "accuracy"],
    )

    erl = EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=3,
        restore_best_weights=True,
    )

    tf_model.fit(
        data.x_train,
        data.y_train,
        epochs=tf_params["epochs"],
        batch_size=tf_params["batch_size"],
        callbacks=[erl],
        validation_data=(data.x_test, data.y_test),
        class_weight=data.class_weight,
        verbose=1,
    )
    _, tf_val_auc, _ = tf_model.evaluate(data.x_test, data.y_test, verbose=0)
    print(f"  ✔ best val_auc: {tf_val_auc:.4f}")
    tf_results.append({**tf_params, "val_auc": tf_val_auc})
    if tf_val_auc > tf_global_best["auc"]:
        tf_global_best["auc"] = tf_val_auc
        tf_model.save(TF_MODEL_PATH)

tf_df = (
    pd.DataFrame(tf_results)
    .sort_values("val_auc", ascending=False)
    .reset_index(drop=True)
)
tf_df.index += 1
print(f"\n{'═'*70}\n{tf_df.to_string()}\n{'═'*70}")

tf_best_model = tf.keras.models.load_model(TF_MODEL_PATH)
_, tf_auc, tf_acc = tf_best_model.evaluate(data.x_test, data.y_test, verbose=0)
print(f"\nTensorFlow best model  —  test AUC: {tf_auc:.4f}  |  accuracy: {tf_acc:.4f}")



[1/8] {'learning_rate': 0.001, 'dropout_rate': 0.1, 'l2_reg': 1e-06, 'batch_size': 256, 'epochs': 200, 'hidden_units': 512}
Epoch 1/200
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 19s 15ms/step - accuracy: 0.6453 - auc: 0.7777 - loss: 0.5664 - val_accuracy: 0.6887 - val_auc: 0.8060 - val_loss: 0.5248
Epoch 2/200
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - accuracy: 0.6799 - auc: 0.8162 - loss: 0.5220 - val_accuracy: 0.7046 - val_auc: 0.8190 - val_loss: 0.4717
Epoch 3/200
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - accuracy: 0.6883 - auc: 0.8251 - loss: 0.5105 - val_accuracy: 0.6597 - val_auc: 0.8235 - val_loss: 0.5614
Epoch 4/200
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - accuracy: 0.6975 - auc: 0.8328 - loss: 0.5006 - val_accuracy: 0.7043 - val_auc: 0.8259 - val_loss: 0.5029
Epoch 5/200
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - accuracy: 0.7034 - auc: 0.8363 - loss: 0.4963 - val_accuracy: 0.6708 - val_auc: 0.8274 - val_loss: 0.5698
Epoch 6/200
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 16s 14m

## 3. Predictions

Generate predictions on the holdout test set using the best neural network model.

In [12]:
from features import get_test_data

selected_features = data.x_train.columns.tolist()
df_test = get_test_data("../data/bank_data_test.csv", data)
test_features = df_test[selected_features]

# Pick the model with the higher test AUC
best_nn_model = keras_best_model if keras_auc >= tf_auc else tf_best_model
print(
    f"Using {'Keras' if keras_auc >= tf_auc else 'TensorFlow'} model (AUC: {max(keras_auc, tf_auc):.4f})"
)

predictions = best_nn_model.predict(test_features, verbose=0)
predictions[:5]

Using TensorFlow model (AUC: 0.8355)


array([[0.6659443 ],
       [0.6310384 ],
       [0.07139289],
       [0.22145869],
       [0.00778314]], dtype=float32)

In [13]:
output_df = pd.DataFrame({
    'ID':     df_test['ID'],
    'TARGET': predictions.flatten(),
})
os.makedirs("../predictions", exist_ok=True)
output_df.to_csv("../predictions/churn_keras_predictions.csv", index=False)
print(f"Saved {len(output_df)} predictions to ../predictions/churn_keras_predictions.csv")
output_df.head()

Saved 88798 predictions to ../predictions/churn_keras_predictions.csv


,ID,TARGET
0,400980,0.665944
1,525062,0.631038
2,280316,0.071393
3,496066,0.221459
4,375031,0.007783


## 4. Results Summary

Comparison of all models: library, algorithm, key hyperparameters, accuracy and AUC.

In [15]:
results_table = pd.DataFrame(
    [
        {
            "Algorithm": "Naive Classifier (Most Frequent)",
            "Accuracy": round(naive_acc, 4),
            "AUC": round(naive_auc, 4),
            "Library": "scikit-learn",
            "Hyperparameters": "strategy=most_frequent",
        },
        {
            "Algorithm": "Random Forest",
            "Accuracy": round(rf_acc, 4),
            "AUC": round(rf_auc, 4),
            "Library": "scikit-learn",
            "Hyperparameters": (
                f"n_estimators={rf_best_params['n_estimators']}, "
                f"max_depth={rf_best_params['max_depth']}, "
                f"min_samples_split={rf_best_params['min_samples_split']}, "
                f"min_samples_leaf={rf_best_params['min_samples_leaf']}"
            ),
        },
        {
            "Algorithm": "MLPClassifier",
            "Accuracy": round(mlp_acc, 4),
            "AUC": round(mlp_auc, 4),
            "Library": "scikit-learn",
            "Hyperparameters": (
                f"hidden_layers={MLP_HIDDEN}, " f"lr={MLP_LR}, max_iter={MLP_MAX_ITER}"
            ),
        },
        {
            "Algorithm": "Neural Network",
            "Accuracy": round(keras_acc, 4),
            "AUC": round(keras_auc, 4),
            "Library": "Keras (TensorFlow)",
            "Hyperparameters": (
                f"lr={keras_params['learning_rate']}, "
                f"dropout={keras_params['dropout_rate']}, "
                f"l2={keras_params['l2_reg']}, "
                f"batch={keras_params['batch_size']}, "
                f"epochs={keras_params['epochs']}, "
                f"units={keras_params['hidden_units']}"
            ),
        },
        {
            "Algorithm": "Neural Network",
            "Accuracy": round(tf_acc, 4),
            "AUC": round(tf_auc, 4),
            "Library": "TensorFlow",
            "Hyperparameters": (
                f"lr={tf_params['learning_rate']}, "
                f"dropout={tf_params['dropout_rate']}, "
                f"l2={tf_params['l2_reg']}, "
                f"batch={tf_params['batch_size']}, "
                f"epochs={tf_params['epochs']}, "
                f"units={tf_params['hidden_units']}"
            ),
        },
        {
            "Algorithm": "Neural Network",
            "Accuracy": round(numpy_acc, 4),
            "AUC": round(numpy_auc, 4),
            "Library": "NumPy (custom)",
            "Hyperparameters": (
                f"hidden_layers={NUMPY_HIDDEN}, "
                f"lr={NUMPY_LR}, "
                f"epochs={NUMPY_EPOCHS}, "
                f"batch={NUMPY_BATCH}"
            ),
        },
    ]
)

results_table.index += 1
# display all columns and full cell content without truncation
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
results_table


,Algorithm,Accuracy,AUC,Library,Hyperparameters
1,Naive Classifier (Most Frequent),0.9186,0.5000,scikit-learn,strategy=most_frequent
2,Random Forest,0.9194,0.8353,scikit-learn,"n_estimators=200, max_depth=20, min_samples_split=2, min_samples_leaf=2"
3,MLPClassifier,0.9174,0.8294,scikit-learn,"hidden_layers=(512, 256, 128), lr=0.001, max_iter=10"
4,Neural Network,0.7117,0.8321,Keras (TensorFlow),"lr=0.001, dropout=0.1, l2=1e-06, batch=256, epochs=200, units=512"
5,Neural Network,0.7251,0.8355,TensorFlow,"lr=0.0005, dropout=0.3, l2=0.0001, batch=256, epochs=200, units=512"
6,Neural Network,0.7045,0.8056,NumPy (custom),"hidden_layers=[512, 256, 128], lr=0.001, epochs=60, batch=256"
